# Week 2 — Exploratory Data Analysis and Data Preprocessing

**DATS 511 — Applied Statistics and Data Analysis · Fall 2026**  
This notebook is an **in-class study file**. Work through the cells in order, discuss the statistical choices, and compare the raw and cleaned data.

[Open in Google Colab](https://colab.research.google.com/github/omzeybek/Yeditepe_Data_Science_DATS511/blob/main/Week2_EDA.ipynb)

We use **554 entirely fictional individuals**, localized for Türkiye. The observations are cross-sectional; they do not describe real people or a representative Turkish population. Wages are monthly TRY, wealth is TRY, height is cm, weight is kg, and home area is m². YKS values are simulated scores rather than historical exam results.

The source brief literally requests **0.05% missing fields** and **0.02% injected numeric outliers**. Rounded to whole cells, that gives five missing fields and one intentionally erroneous education-years value of 1,000. Naturally occurring IQR outliers can be more numerous. Keep that distinction throughout the analysis.

**Workflow:** import packages → create/load the data in Drive → inspect and visualize raw data → diagnose data quality → remove the measurement-error record → impute selected fields → compare raw and log-transformed wealth.

On Colab, the Drive cell asks you to authorize your own Google account and creates the dataset only if it is absent. In a local Jupyter environment, the notebook loads the accompanying `data/week2_personal_data.csv`.

## 1. Packages and reproducibility

| Package | Role |
|---|---|
| NumPy | Seeded random generation and numerical operations |
| pandas | DataFrames, summaries, frequency tables, and CSV files |
| Matplotlib | Figure layout, annotation, and plot control |
| seaborn | Statistical plots with categorical grouping |
| scikit-learn | Standardization and nearest-neighbor imputation |
| pathlib / json | File paths and dataset metadata |

Colab normally provides these packages. If an import fails in a different environment, install the missing package with `%pip install package_name`, then restart the kernel. The repository README records the versions used for validation.

In [ ]:
%matplotlib inline
import os
import json
from pathlib import Path
from importlib.metadata import version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

SEED = 5112026
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 20)
numeric_cols = ["age", "weight_kg", "height_cm", "education_years",
                "monthly_wage_try", "wealth_try", "home_sqm", "yks_score"]
education_order = ["İlkokul", "Ortaokul", "Lise", "Ön Lisans", "Lisans", "Yüksek Lisans", "Doktora"]
display(pd.Series({name: version(name) for name in
                   ["numpy", "pandas", "matplotlib", "seaborn", "scikit-learn"]}, name="installed_version"))

## 2. Create the reproducible synthetic dataset

The generator below is included so the notebook can create its own CSV inside your mounted Drive. It uses a fixed seed, fictional `SYN…` identifiers, plausible education/age relationships, and a dispersed positive wealth distribution. At least one legitimate doctoral record has 25 years of education.

Missing values are deliberately placed in wage, wealth, height, weight, and home area so each requested imputation can be demonstrated even at the very small literal missingness rate. This is a teaching design, not a claim that real missingness is random. The separately injected 1,000-year measurement error must not be mistaken for a plausible tail observation.

In [ ]:
"""Reproducible fictional Turkish-person data for a DATS 511 classroom exercise."""
import numpy as np
import pandas as pd

COLUMNS = [
    "name", "surname", "person_id", "age", "occupation", "sex", "weight_kg",
    "height_cm", "education_years", "education_level", "monthly_wage_try",
    "wealth_try", "home_sqm", "city", "yks_score", "university_completed",
    "department_completed",
]
NUMERIC_COLUMNS = [
    "age", "weight_kg", "height_cm", "education_years", "monthly_wage_try",
    "wealth_try", "home_sqm", "yks_score",
]
EDUCATION_ORDER = ["İlkokul", "Ortaokul", "Lise", "Ön Lisans", "Lisans", "Yüksek Lisans", "Doktora"]


def generate_dataset(missing_rate=0.0005, outlier_rate=0.0002, seed=5112026):
    """Return a 554-row DataFrame and a JSON-serializable provenance dictionary.

    Rates use all 17 columns for missingness and the 8 numeric columns for
    outliers. Thus 0.05% / 0.02% produce five missing cells / one error.
    Set 0.05 / 0.02 for the alternative 5% / 2% interpretation.
    Missingness is inserted only in numeric columns: the five variables
    requested for imputation are prioritized, then other numeric cells.
    """
    if not (0 <= missing_rate <= 1 and 0 <= outlier_rate <= 1):
        raise ValueError("Rates must be proportions between zero and one.")
    rng = np.random.default_rng(seed)
    n = 554
    sex = rng.choice(["Kadın", "Erkek"], n)
    female_names = ["Ayşe", "Fatma", "Zeynep", "Elif", "Merve", "Esra", "Derya", "Selin", "Ece", "İrem", "Özge", "Deniz"]
    male_names = ["Mehmet", "Mustafa", "Ahmet", "Ali", "Emre", "Burak", "Murat", "Can", "Kerem", "Oğuz", "Uğur", "Deniz"]
    names = [rng.choice(female_names if s == "Kadın" else male_names) for s in sex]
    surnames = rng.choice(["Yılmaz", "Kaya", "Demir", "Çelik", "Şahin", "Yıldız", "Öztürk", "Aydın", "Arslan", "Doğan", "Kılıç", "Koç", "Kurt", "Aksoy", "Güneş", "Özdemir"], n)
    level = rng.choice(EDUCATION_ORDER, n, p=[0.07, 0.08, 0.30, 0.13, 0.28, 0.10, 0.04])
    years_by_level = dict(zip(EDUCATION_ORDER, [5, 8, 12, 14, 16, 18, 23]))
    years = np.array([years_by_level[x] for x in level])
    doctorate = np.flatnonzero(level == "Doktora")
    years[doctorate] = rng.integers(22, 26, len(doctorate))
    level[0], years[0] = "Doktora", 25  # A valid long doctoral education.
    minimum_age = np.maximum(18, years + 6)
    age = np.maximum(rng.integers(18, 76, n), minimum_age)
    age[0] = 43
    jobs = {
        "İlkokul": ["İşçi", "Çiftçi", "Esnaf", "Ev işleriyle meşgul"],
        "Ortaokul": ["İşçi", "Şoför", "Esnaf", "Satış danışmanı"],
        "Lise": ["Satış danışmanı", "İdari personel", "Şoför", "İşletme sahibi"],
        "Ön Lisans": ["Tekniker", "Sağlık teknikeri", "İdari personel"],
        "Lisans": ["Mühendis", "Öğretmen", "Muhasebeci", "Yazılım geliştirici"],
        "Yüksek Lisans": ["Mühendis", "Yönetici", "Yazılım geliştirici", "Öğretmen"],
        "Doktora": ["Akademisyen", "Araştırmacı", "Yönetici"],
    }
    occupation = [rng.choice(jobs[x]) for x in level]
    for i in range(1, n):
        if age[i] >= 60 and rng.random() < 0.65:
            occupation[i] = "Emekli"
        elif age[i] < 23 and level[i] == "Lise" and rng.random() < 0.35:
            occupation[i] = "Öğrenci"
        elif rng.random() < 0.035:
            occupation[i] = "İşsiz"
    occupation[0] = "Akademisyen"
    height = np.clip(rng.normal(np.where(sex == "Kadın", 163, 176), 6.5), 143, 198)
    bmi = np.clip(rng.normal(25.5 + 0.018 * (age - 35), 3.2), 18.5, 36)
    weight = np.clip(bmi * (height / 100) ** 2, 43, 125)
    wage_base = dict(zip(EDUCATION_ORDER, [29000, 31000, 36000, 44000, 61000, 79000, 103000]))
    wage = np.array([wage_base[x] for x in level]) * rng.lognormal(0, 0.30, n)
    wage *= 0.85 + np.minimum(age - 18, 30) / 100
    for i, job in enumerate(occupation):
        if job == "Emekli":
            wage[i] *= 0.55
        elif job in ["Öğrenci", "İşsiz", "Ev işleriyle meşgul"]:
            wage[i] = 0
    # Strictly positive and strongly right-skewed: useful for a log-transform.
    wealth = rng.lognormal(np.log(2200000), 1.60, n) * (0.55 + age / 60)
    home = np.clip(95 + 20 * np.log1p(wealth / 2200000) + rng.normal(0, 25, n), 30, 430)
    cities = rng.choice(["İstanbul", "Ankara", "İzmir", "Bursa", "Antalya", "Adana", "Konya", "Gaziantep", "Eskişehir", "Samsun", "Trabzon", "Diyarbakır", "Kayseri", "Mersin"], n)
    universities = ["Yeditepe Üniversitesi", "İstanbul Üniversitesi", "Ankara Üniversitesi", "Ege Üniversitesi", "Hacettepe Üniversitesi", "Marmara Üniversitesi", "Dokuz Eylül Üniversitesi", "Orta Doğu Teknik Üniversitesi", "Anadolu Üniversitesi"]
    departments = {
        "Ön Lisans": ["Bilgisayar Programcılığı", "İşletme Yönetimi", "Tıbbi Laboratuvar Teknikleri"],
        "Lisans": ["Bilgisayar Mühendisliği", "İşletme", "İstatistik", "Matematik", "İktisat", "Sınıf Öğretmenliği"],
        "Yüksek Lisans": ["Veri Bilimi", "İşletme", "İstatistik", "Bilgisayar Mühendisliği", "İktisat"],
        "Doktora": ["İstatistik", "Bilgisayar Mühendisliği", "İktisat", "İşletme"],
    }
    # Job and qualification labels refer to broad education groups, not a
    # specific verified professional license or a person's real credentials.
    university = [rng.choice(universities) if x in departments else "Üniversite tamamlanmadı" for x in level]
    department = [rng.choice(departments[x]) if x in departments else "Bölüm tamamlanmadı" for x in level]
    yks = np.clip(230 + 13 * (years - 12) + rng.normal(0, 45, n), 150, 500)
    df = pd.DataFrame({
        "name": names, "surname": surnames,
        "person_id": [f"SYN{i + 1:04d}" for i in range(n)], "age": age,
        "occupation": occupation, "sex": sex, "weight_kg": np.round(weight, 1),
        "height_cm": np.round(height, 1), "education_years": years,
        "education_level": level, "monthly_wage_try": np.round(wage, 2),
        "wealth_try": np.round(wealth, 2), "home_sqm": np.round(home, 1),
        "city": cities, "yks_score": np.round(yks, 1),
        "university_completed": university, "department_completed": department,
    }, columns=COLUMNS)
    missing_target = int(round(n * len(COLUMNS) * missing_rate))
    outlier_target = int(round(n * len(NUMERIC_COLUMNS) * outlier_rate))
    injected = []
    protected = {(0, "education_years")}  # Retain the valid PhD example.
    if outlier_target:
        error_row = int(rng.integers(1, n))
        old = float(df.at[error_row, "education_years"])
        df.at[error_row, "education_years"] = 1000
        injected.append({"row_index": error_row, "person_id": df.at[error_row, "person_id"], "column": "education_years", "original_value": old, "injected_value": 1000, "kind": "measurement_error", "reason": "Impossible education duration; flag for explicit measurement-error handling."})
        protected.add((error_row, "education_years"))
    # At the higher optional rate, add plausible extreme financial values;
    # these are contextual outliers, not automatically measurement errors.
    eligible = [(i, col) for i in range(1, n) for col in ["wealth_try", "home_sqm", "monthly_wage_try"] if col != "monthly_wage_try" or occupation[i] not in ["Emekli", "Öğrenci", "İşsiz", "Ev işleriyle meşgul"]]
    if outlier_target - 1 > len(eligible):
        raise ValueError("Outlier rate exceeds available contextual-outlier cells.")
    if outlier_target > 1:
        for pos in rng.choice(len(eligible), size=outlier_target - 1, replace=False):
            row, col = eligible[int(pos)]
            old = float(df.at[row, col])
            limits = {"wealth_try": (120000000, 600000000), "monthly_wage_try": (450000, 850000), "home_sqm": (550, 1100)}
            new = round(float(rng.uniform(*limits[col])), 2 if col != "home_sqm" else 1)
            df.at[row, col] = new
            injected.append({"row_index": row, "person_id": df.at[row, "person_id"], "column": col, "original_value": old, "injected_value": new, "kind": "contextual_extreme", "reason": "Simulated high financial value, not an impossible measurement."})
            protected.add((row, col))
    missing = []
    priority = ["monthly_wage_try", "wealth_try", "height_cm", "weight_kg", "home_sqm"]
    for col in priority[:missing_target]:
        available = [i for i in range(1, n) if (i, col) not in protected and i not in [x[0] for x in missing]]
        row = int(rng.choice(available))
        missing.append((row, col))
    candidates = [(i, col) for i in range(n) for col in NUMERIC_COLUMNS if (i, col) not in protected and (i, col) not in missing]
    extra = missing_target - len(missing)
    if extra > len(candidates):
        raise ValueError("Missing rate exceeds available numeric cells.")
    if extra:
        missing.extend(candidates[int(p)] for p in rng.choice(len(candidates), extra, replace=False))
    missing_records = []
    for row, col in missing:
        missing_records.append({"row_index": row, "person_id": df.at[row, "person_id"], "column": col, "original_value": float(df.at[row, col])})
        df.at[row, col] = np.nan
    injected_pairs = {(x["row_index"], x["column"]) for x in injected}
    iqr_flags = {}
    for col in NUMERIC_COLUMNS:
        q1, q3 = df[col].quantile([0.25, 0.75])
        lower, upper = float(q1 - 1.5 * (q3 - q1)), float(q3 + 1.5 * (q3 - q1))
        rows = df.index[(df[col] < lower) | (df[col] > upper)].tolist()
        iqr_flags[col] = {"lower_fence": lower, "upper_fence": upper, "flagged_count": len(rows), "naturally_generated_rows": [int(i) for i in rows if (i, col) not in injected_pairs], "injected_rows": [int(i) for i in rows if (i, col) in injected_pairs]}
    metadata = {
        "synthetic": True, "disclosure": "All names, identifiers and measurements are fictional; this is a teaching sample, not representative population or current official economic data.",
        "seed": int(seed), "rows": n, "columns": COLUMNS, "numeric_columns": NUMERIC_COLUMNS,
        "education_order": EDUCATION_ORDER,
        "units": {"age": "years", "weight_kg": "kg", "height_cm": "cm", "education_years": "years", "monthly_wage_try": "TRY/month; zero for no wage", "wealth_try": "TRY; positive synthetic total wealth", "home_sqm": "m²", "yks_score": "simulated 150–500 YKS-scale score"},
        "yks_score_note": "Scores are fictional YKS-scale proxies for all adults, not historical exam results; older cohorts did not take YKS under its current name.",
        "missing_rate_requested": float(missing_rate), "missing_denominator": n * len(COLUMNS), "missing_count": len(missing), "missing_rate_actual": len(missing) / (n * len(COLUMNS)), "missing_locations": missing_records,
        "outlier_rate_requested": float(outlier_rate), "outlier_denominator": n * len(NUMERIC_COLUMNS), "injected_outlier_count": len(injected), "injected_rate_actual": len(injected) / (n * len(NUMERIC_COLUMNS)), "injected_outliers": injected,
        "outlier_note": "Requested outlier rate controls deliberate injections; natural values flagged by the IQR rule are recorded separately and may be valid.",
        "valid_phd_example": {"row_index": 0, "person_id": "SYN0001", "age": 43, "education_years": 25},
        "iqr_outliers": iqr_flags,
    }
    return df, metadata


In [ ]:
generated_df, dataset_metadata = generate_dataset(
    missing_rate=0.0005, outlier_rate=0.0002, seed=SEED
)
print("Generated shape:", generated_df.shape)
print("Missing fields:", int(generated_df.isna().sum().sum()))
print("Injected measurement errors in education_years:", int((generated_df["education_years"] == 1000).sum()))

## 3. Mount Drive, persist the CSV, and load a DataFrame

In Colab, authorize the Drive prompt. Files live in `MyDrive/DATS511/Week2/`. If you prefer to upload the GitHub CSV manually, place it in that folder before running this cell. Otherwise the generator supplies the identical sample automatically. Existing files are preserved.

Locally, use the repository's `data/` directory. `DATS511_DATA_PATH` and `DATS511_OUTPUT_DIR` are optional local path overrides. Analysis exports go to a separate output folder.

In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    DATA_DIR = Path("/content/drive/MyDrive/DATS511/Week2")
else:
    DATA_DIR = Path.cwd() / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = Path(os.environ.get("DATS511_DATA_PATH", str(DATA_DIR / "week2_personal_data.csv")))
META_PATH = DATA_PATH.with_name("week2_dataset_metadata.json")

if not DATA_PATH.exists():
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    generated_df.to_csv(DATA_PATH, index=False, encoding="utf-8")
    META_PATH.write_text(json.dumps(dataset_metadata, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    print("Created synthetic dataset:", DATA_PATH)
else:
    print("Using existing dataset:", DATA_PATH)

df = pd.read_csv(DATA_PATH, encoding="utf-8", dtype={"person_id": "string"})
metadata = json.loads(META_PATH.read_text(encoding="utf-8")) if META_PATH.exists() else {}
assert df.shape == (554, 17), "This lesson expects the supplied 554 × 17 synthetic dataset."
assert df["person_id"].is_unique, "Each synthetic person must have a unique identifier."
default_output_dir = DATA_DIR / "outputs" if IN_COLAB else Path.cwd() / "outputs"
OUTPUT_DIR = Path(os.environ.get("DATS511_OUTPUT_DIR", str(default_output_dir)))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
display(df.head())

## 4. Explore the raw data

These descriptive analyses use the original synthetic data, including missing values and the deliberately implausible **1,000 years of education**. Keeping that value visible lets us see why a later quality check matters. No rows are removed or imputed in this section.

The units are age and education in years; weight in kg; height in cm; monthly wage in TRY/month; wealth in TRY; home area in m²; and a fictional YKS score simulated between 150 and 500 for these adults. These scores are an exercise variable, not historical examination results. Wages and wealth are nominal simulated TRY amounts, with no implied valuation date or purchasing-power adjustment.

Each table states its denominator. Associations describe this generated sample; they do not establish population estimates or causes. Names and identifiers are labels rather than quantitative measurements.

In [ ]:
import io
import unicodedata
from matplotlib.ticker import FuncFormatter, MaxNLocator

# Work with numeric views without changing the original dataframe.
raw_numeric = df[numeric_cols].apply(pd.to_numeric, errors="coerce")
units = {
    "age": "years", "weight_kg": "kg", "height_cm": "cm",
    "education_years": "years", "monthly_wage_try": "TRY/month",
    "wealth_try": "TRY", "home_sqm": "m²", "yks_score": "points",
}
titles = {
    "age": "Age", "weight_kg": "Weight", "height_cm": "Height",
    "education_years": "Years of education", "monthly_wage_try": "Monthly wage",
    "wealth_try": "Wealth", "home_sqm": "Home area", "yks_score": "Fictional YKS score",
}
number_formatter = FuncFormatter(lambda value, position: f"{value:,.0f}")
categorical_cols = [column for column in df.columns if column not in numeric_cols]

print(f"Raw shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
schema = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_missing": df.notna().sum(),
    "missing": df.isna().sum(),
    "distinct_non_missing": df.nunique(dropna=True),
})
schema["missing_percent"] = 100 * schema["missing"] / max(len(df), 1)
display(schema)
info_buffer = io.StringIO()
df.info(buf=info_buffer)
print(info_buffer.getvalue())

### Raw descriptive statistics and quick numeric insights

`describe(include="all")` summarizes every variable. Numeric quartiles and standard deviations have measurement units; categorical `top` and `freq` identify the most frequent label. An identifier's most frequent value is not a substantive population characteristic.

The additional numeric table compares means with medians and counts Tukey outliers, using fences **Q1 − 1.5 × IQR** and **Q3 + 1.5 × IQR**. These fences flag unusual observations; an outlier is not automatically an error. The implausible education value is evaluated separately during data-quality checks. Missing values are excluded separately for each numeric summary.

In [ ]:
display(df.describe(include="all").T)

def numeric_summary(series):
    values = pd.to_numeric(series, errors="coerce").dropna()
    if values.empty:
        return pd.Series({"n": 0, "mean": np.nan, "median": np.nan, "Q1": np.nan,
                          "Q3": np.nan, "IQR": np.nan, "min": np.nan, "max": np.nan,
                          "Tukey_outliers": 0, "outlier_percent_of_observed": np.nan})
    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1
    outliers = (values < q1 - 1.5 * iqr) | (values > q3 + 1.5 * iqr)
    return pd.Series({"n": len(values), "mean": values.mean(), "median": values.median(),
                      "Q1": q1, "Q3": q3, "IQR": iqr, "min": values.min(),
                      "max": values.max(), "Tukey_outliers": int(outliers.sum()),
                      "outlier_percent_of_observed": 100 * outliers.mean()})

raw_numeric_summary = raw_numeric.apply(numeric_summary).T
raw_numeric_summary["unit"] = pd.Series(units)
display(raw_numeric_summary.round(2))

print("Descriptive observations from the generated sample:")
for column in ["age", "monthly_wage_try", "wealth_try", "education_years"]:
    row = raw_numeric_summary.loc[column]
    if row["n"]:
        print(f"• {titles[column]}: median {row['median']:,.2f}, mean {row['mean']:,.2f} "
              f"{units[column]}; {int(row['Tukey_outliers'])} of {int(row['n'])} observed values "
              "lie beyond the IQR fences.")
print("Mean–median differences can reflect skewness or influential extremes; they do not identify a cause.")
print("The 1,000-year education observation remains in these raw summaries and can strongly affect the mean.")
print("Synthetic, descriptive results are neither causal evidence nor estimates for Türkiye's population.")

### Categorical frequencies: nominal labels and ordered education

Names, surnames, identifiers, sex, occupation, city, university, and department are nominal labels. Education level is ordinal, so its table follows the specified education order. Its category codes must not be treated as distances or continuous measurements.

The tables include missing values and divide by **all rows**. For columns with many distinct labels, only the ten most frequent rows are displayed, with a combined remainder that keeps the denominator complete. Names and IDs are summarized for inspection but are not plotted.

In [ ]:
MISSING_LABEL = "(Missing)"

def as_labels(series):
    return series.astype("string").fillna(MISSING_LABEL)

def category_frequency(series, order=None, top_n=None):
    counts = as_labels(series).value_counts(dropna=False)
    if order is not None:
        remaining = [label for label in counts.index if label not in order]
        counts = counts.reindex(list(order) + remaining, fill_value=0)
    elif top_n is not None and len(counts) > top_n:
        remainder = int(counts.iloc[top_n:].sum())
        counts = counts.iloc[:top_n].copy()
        counts.loc[f"(Other {len(as_labels(series).unique()) - top_n} labels combined)"] = remainder
    table = counts.rename("count").to_frame()
    table["relative_frequency"] = table["count"] / max(len(series), 1)
    table["percent_of_all_rows"] = 100 * table["relative_frequency"]
    return table

for column in categorical_cols:
    print(f"\n{column}: {df[column].nunique(dropna=True)} distinct observed labels; denominator = {len(df):,} rows")
    display(category_frequency(df[column], order=education_order if column == "education_level" else None,
                               top_n=None if column == "education_level" else 10).round(4))

### Boxplots for all eight numeric variables

Each panel uses its own scale and units. A blue line marks the median, a green diamond the mean, and purple × marks up to three tied modes when values repeat. For a continuously measured variable, a mode may simply reflect rounding; if every value occurs once, there is no informative empirical mode.

The annotation reports mean, median, mode, IQR, and the **total** number of observations beyond the Tukey fences. All flagged points are shown; labels identify at most the lowest and highest flagged observations to keep the plot readable. The raw 1,000-year education value is intentionally retained, so that panel's central distribution is compressed.

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(15, 19), constrained_layout=True)
for ax, column in zip(axes.flat, numeric_cols):
    values = raw_numeric[column].dropna()
    ax.set_title(f"{titles[column]} — {units[column]}")
    if values.empty:
        ax.text(0.5, 0.5, "No observed numeric values", transform=ax.transAxes, ha="center")
        continue
    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    flagged = values[(values < lower) | (values > upper)]
    counts = values.value_counts()
    modes = counts[counts == counts.max()].index.sort_values() if counts.max() > 1 else []
    mode_text = ", ".join(f"{value:,.2f}" for value in modes[:3]) if len(modes) else "no repeated value"
    if len(modes) > 3:
        mode_text += f" (+{len(modes) - 3} tied)"
    ax.boxplot(values.to_numpy(), positions=[1], widths=0.45, showfliers=False,
               medianprops={"color": "royalblue", "linewidth": 2},
               boxprops={"color": "#444444"}, whiskerprops={"color": "#444444"})
    ax.scatter([1], [values.mean()], marker="D", color="seagreen", s=48, label="Mean", zorder=4)
    ax.scatter([1] * min(len(modes), 3), list(modes[:3]), marker="x", color="purple",
               s=65, label="Mode (up to 3)", zorder=4)
    if len(flagged):
        jitter = np.linspace(-0.10, 0.10, len(flagged)) if len(flagged) > 1 else np.array([0.0])
        ax.scatter(1 + jitter, flagged.to_numpy(), s=24, alpha=0.7, color="crimson",
                   label="Beyond IQR fences", zorder=3)
        extremes = sorted(set([float(flagged.min()), float(flagged.max())]))
        for position, extreme in enumerate(extremes):
            ax.annotate(f"{extreme:,.2f}", xy=(1, extreme),
                        xytext=(35, 10 if position else -10), textcoords="offset points",
                        fontsize=8, color="crimson", arrowprops={"arrowstyle": "-", "color": "crimson"})
    ax.text(0.02, 0.98,
            f"n = {len(values):,} | mean = {values.mean():,.2f}\n"
            f"median = {values.median():,.2f}\nmode = {mode_text}\n"
            f"IQR = {iqr:,.2f}\nTukey outliers = {len(flagged):,} ({100 * len(flagged) / len(values):.1f}%)",
            transform=ax.transAxes, va="top", fontsize=8,
            bbox={"facecolor": "white", "alpha": 0.9, "edgecolor": "#dddddd"})
    ax.set_xticks([])
    ax.set_xlim(0.5, 1.75)
    ax.set_ylabel(units[column])
    ax.yaxis.set_major_formatter(number_formatter)
    ax.grid(axis="y", alpha=0.2)
fig.suptitle("Raw numeric boxplots: individual scales, all observed values retained", fontsize=15)
plt.show()
plt.close(fig)

### Histograms for all eight numeric variables

Histogram positions represent measured quantities, unlike the category bars below. Each panel includes every observed value. Counts use bins chosen from the observed range: discrete unit-width bins when suitable, otherwise a capped Freedman–Diaconis suggestion. Bin choices affect apparent shape, so a peak is not automatically a distinct group.

Dashed lines mark the mean and median. The raw education extreme creates a long range; it is visible here before cleaning rather than silently clipped.

In [ ]:
def practical_histogram_bins(values, cap=35):
    data = np.asarray(values, dtype=float)
    if data.size == 0:
        return 10
    if np.isclose(data.min(), data.max()):
        return np.array([data.min() - 0.5, data.max() + 0.5])
    integer_data = np.all(np.isclose(data, np.round(data)))
    if integer_data and data.max() - data.min() <= cap:
        return np.arange(np.floor(data.min()) - 0.5, np.ceil(data.max()) + 1.5, 1)
    suggested = np.histogram_bin_edges(data, bins="fd")
    n_bins = min(cap, max(10, len(suggested) - 1))
    return np.linspace(data.min(), data.max(), n_bins + 1)

fig, axes = plt.subplots(4, 2, figsize=(15, 17), constrained_layout=True)
for ax, column in zip(axes.flat, numeric_cols):
    values = raw_numeric[column].dropna()
    ax.set_title(f"{titles[column]} (n = {len(values):,})")
    if values.empty:
        ax.text(0.5, 0.5, "No observed numeric values", transform=ax.transAxes, ha="center")
        continue
    ax.hist(values, bins=practical_histogram_bins(values), color="#397fa8", edgecolor="white", alpha=0.85)
    ax.axvline(values.mean(), color="seagreen", linestyle="--", label="Mean")
    ax.axvline(values.median(), color="darkorange", linestyle=":", label="Median")
    ax.set_xlabel(units[column])
    ax.set_ylabel("Observed count")
    ax.xaxis.set_major_formatter(number_formatter)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.grid(axis="y", alpha=0.2)
    ax.legend(fontsize=8)
fig.suptitle("Raw numeric histograms: all observed values retained", fontsize=15)
plt.show()
plt.close(fig)

### Monthly wage and ordered education level

The grouped boxplot compares wage distributions across observed education labels; the stacked histogram shows how each label contributes to the same wage bins. Both use **wage measurements** on the numeric axis and labels in their education order. Missing or unexpected education labels are retained as separate groups.

These are counts, so larger groups contribute more. Differences may reflect how the fictional data were generated, group sizes, or other variables; they do not estimate the wage effect of obtaining a degree.

In [ ]:
education_labels = as_labels(df["education_level"])
education_groups = list(education_order) + [label for label in education_labels.unique() if label not in education_order]
wage_groups = [raw_numeric.loc[education_labels.eq(label), "monthly_wage_try"].dropna() for label in education_groups]
nonempty_education = [(label, values) for label, values in zip(education_groups, wage_groups) if len(values)]
fig, ax = plt.subplots(figsize=(12, 6), constrained_layout=True)
if nonempty_education:
    ax.boxplot([values.to_numpy() for label, values in nonempty_education],
               tick_labels=[f"{label}\n(n={len(values)})" for label, values in nonempty_education],
               showmeans=True, meanprops={"marker": "D", "markerfacecolor": "seagreen", "markeredgecolor": "seagreen"})
else:
    ax.text(0.5, 0.5, "No observed wages", transform=ax.transAxes, ha="center")
ax.set_title("Monthly wage by education level — raw data")
ax.set_ylabel("Monthly wage (TRY/month)")
ax.yaxis.set_major_formatter(number_formatter)
ax.tick_params(axis="x", rotation=25)
ax.grid(axis="y", alpha=0.2)
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(12, 6), constrained_layout=True)
all_wages = raw_numeric["monthly_wage_try"].dropna()
if nonempty_education:
    ax.hist([values.to_numpy() for label, values in nonempty_education],
            bins=practical_histogram_bins(all_wages, cap=25), stacked=True,
            label=[label for label, values in nonempty_education], edgecolor="white", alpha=0.85)
    ax.legend(title="Education level", fontsize=8)
ax.set_title("Monthly wage histogram stacked by education level")
ax.set_xlabel("Monthly wage (TRY/month)")
ax.set_ylabel("Observed count")
ax.xaxis.set_major_formatter(number_formatter)
ax.yaxis.set_major_locator(MaxNLocator(integer=True))
ax.grid(axis="y", alpha=0.2)
plt.show()
plt.close(fig)
print(f"Wage plots omit {int(raw_numeric['monthly_wage_try'].isna().sum())} rows with missing/non-numeric wages.")

### Category probabilities and an age stem-and-leaf display

The education and occupation charts show **empirical category probabilities**, meaning relative frequencies in these rows, including missing labels. They are bar charts, not histograms of continuous category codes. All category probabilities add to one when the dataset is nonempty; observed frequencies are not population probabilities.

The age stem-and-leaf display preserves individual observations: a stem of 2 and leaf of 3 means 23 years. Repeated leaves represent repeated ages. Fractional ages, if present, retain their fractional leaf rather than being silently rounded to an integer.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7), constrained_layout=True)
for ax, column in zip(axes, ["education_level", "occupation"]):
    table = category_frequency(df[column], order=education_order if column == "education_level" else None)
    positions = np.arange(len(table))
    bars = ax.barh(positions, table["relative_frequency"], color="#447e9e")
    ax.set_yticks(positions, table.index)
    ax.invert_yaxis()
    ax.set_xlim(0, max(0.12, table["relative_frequency"].max() * 1.25) if len(table) else 1)
    for bar, probability in zip(bars, table["relative_frequency"]):
        ax.text(bar.get_width() + 0.004, bar.get_y() + bar.get_height() / 2,
                f"{probability:.1%}", va="center", fontsize=9)
    ax.set_xlabel("Empirical probability (count ÷ all rows)")
    ax.set_title(column.replace("_", " ").title())
    ax.grid(axis="x", alpha=0.2)
plt.show()
plt.close(fig)

ages = raw_numeric["age"].dropna().sort_values()
print("Age stem-and-leaf | key: 2 | 3 = 23 years; leaves preserve duplicates")
if ages.empty:
    print("No observed ages.")
else:
    stems = np.floor(ages / 10).astype(int)
    for stem in range(int(stems.min()), int(stems.max()) + 1):
        leaves = ages.loc[stems.eq(stem)] - 10 * stem
        leaf_text = " ".join(f"{leaf:g}" for leaf in leaves)
        print(f"{stem:>3} | {leaf_text}")
print(f"Age display includes {len(ages):,} observed values; {int(raw_numeric['age'].isna().sum())} missing/non-numeric ages omitted.")

### Mirrored sex counts by occupation and completed degree

These population-pyramid-style charts compare **Kadın** and **Erkek** counts within occupation and completed education level. Here “degree” means `education_level`, including pre-university education; `department_completed` is a field of study rather than an ordered degree. The negative coordinates only place women on the left; tick labels show positive counts.

These category pyramids describe sample composition rather than an age structure or a sex-specific rate. Missing or other sex labels, if any, are reported separately because the two-sided display contains these two named categories only.

In [ ]:
sex_labels = as_labels(df["sex"])
sex_order = ["Kadın", "Erkek"]
other_sex = ~sex_labels.isin(sex_order)
print(f"Two-sided sex charts include {int((~other_sex).sum()):,} rows with Kadın/Erkek labels.")
if other_sex.any():
    print("Other/missing sex labels outside the two sides:")
    display(sex_labels.loc[other_sex].value_counts().rename("count").to_frame())

def mirrored_counts(ax, categories, order, title, xlabel="Observed people (absolute counts)"):
    table = pd.crosstab(categories, sex_labels).reindex(index=order, columns=sex_order, fill_value=0)
    y_positions = np.arange(len(table))
    ax.barh(y_positions, -table["Kadın"], color="#a04e89", label="Kadın (left)")
    ax.barh(y_positions, table["Erkek"], color="#447e9e", label="Erkek (right)")
    ax.set_yticks(y_positions, table.index)
    ax.invert_yaxis()
    maximum = max(1, int(table.to_numpy().max()) if table.size else 1)
    ax.set_xlim(-maximum * 1.2, maximum * 1.2)
    ax.axvline(0, color="#555555", linewidth=0.8)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=7))
    ax.xaxis.set_major_formatter(FuncFormatter(lambda value, position: f"{abs(value):,.0f}"))
    ax.set_xlabel(xlabel)
    ax.set_title(title)
    ax.grid(axis="x", alpha=0.2)
    ax.legend(fontsize=8)
    return table

occupation_labels = as_labels(df["occupation"])
occupation_order = occupation_labels.value_counts().index.tolist()
fig, axes = plt.subplots(1, 2, figsize=(17, max(6, len(occupation_order) * 0.36)), constrained_layout=True)
occupation_sex_counts = mirrored_counts(axes[0], occupation_labels, occupation_order, "Occupation: mirrored sex counts")
education_sex_counts = mirrored_counts(axes[1], education_labels, education_groups, "Completed education: mirrored sex counts")
plt.show()
plt.close(fig)

### Home area, occupation, and sex

The stacked histogram places **home area in m²** on its measured numeric axis and stacks occupation categories within each interval. Its boundaries start at natural zero and use 50, 75, 100, 150, and 200 m², plus an upper boundary large enough to include the largest observed home. Negative values, if present, also remain visible. Unequal interval widths make the heights counts per interval, not comparable probability densities per m².

The following faceted pyramids use exhaustive intervals: below 0 (a quality flag), 0–<50, 50–<75, 75–<100, 100–<150, 150–<200, and 200+ m². Women and men are mirrored within each occupation group. If there are more than seven occupations, the six largest are shown separately and all remaining occupations are combined explicitly. Counts omit missing home area and sex labels outside the two named sides; those omissions are reported. Group differences do not show that occupation causes a particular home size.

In [ ]:
home = raw_numeric["home_sqm"]
observed_home = home.dropna()
occupation_counts = occupation_labels.value_counts()
if len(occupation_counts) > 7:
    main_occupations = occupation_counts.index[:6].tolist()
    home_occupation = occupation_labels.where(occupation_labels.isin(main_occupations), "Other occupations (combined)")
    home_occupation_order = main_occupations + ["Other occupations (combined)"]
else:
    home_occupation = occupation_labels.copy()
    home_occupation_order = occupation_counts.index.tolist()

fig, ax = plt.subplots(figsize=(13, 6), constrained_layout=True)
if len(observed_home):
    lower_edge = min(0.0, float(np.floor(observed_home.min() / 50) * 50))
    upper_edge = max(250.0, float((np.floor(observed_home.max() / 50) + 1) * 50))
    home_histogram_edges = np.unique([lower_edge, 0, 50, 75, 100, 150, 200, upper_edge])
    datasets = [home.loc[home_occupation.eq(label)].dropna().to_numpy() for label in home_occupation_order]
    ax.hist(datasets, bins=home_histogram_edges, stacked=True, label=home_occupation_order,
            edgecolor="white", alpha=0.85)
    ax.set_xticks(home_histogram_edges)
    ax.legend(title="Occupation", fontsize=8)
else:
    ax.text(0.5, 0.5, "No observed home areas", transform=ax.transAxes, ha="center")
ax.set_title("Home-area histogram stacked by occupation (unequal-width intervals)")
ax.set_xlabel("Home area (m²)")
ax.set_ylabel("Observed count per interval")
ax.yaxis.set_major_locator(MaxNLocator(integer=True))
ax.grid(axis="y", alpha=0.2)
plt.show()
plt.close(fig)

home_bin_labels = ["Below 0 (check)", "0–<50", "50–<75", "75–<100", "100–<150", "150–<200", "200+"]
home_bins = pd.cut(home, bins=[-np.inf, 0, 50, 75, 100, 150, 200, np.inf],
                   labels=home_bin_labels, right=False, include_lowest=True)
n_panels = max(1, len(home_occupation_order))
n_columns = 2
n_rows = (n_panels + n_columns - 1) // n_columns
fig, axes = plt.subplots(n_rows, n_columns, figsize=(15, 4 * n_rows), squeeze=False, constrained_layout=True)
for ax, occupation in zip(axes.flat, home_occupation_order):
    selection = home_occupation.eq(occupation) & home.notna()
    counts = pd.crosstab(home_bins.loc[selection], sex_labels.loc[selection], dropna=False)
    counts = counts.reindex(index=home_bin_labels, columns=sex_order, fill_value=0)
    y_positions = np.arange(len(home_bin_labels))
    ax.barh(y_positions, -counts["Kadın"], color="#a04e89", label="Kadın (left)")
    ax.barh(y_positions, counts["Erkek"], color="#447e9e", label="Erkek (right)")
    ax.set_yticks(y_positions, home_bin_labels)
    ax.invert_yaxis()
    maximum = max(1, int(counts.to_numpy().max()) if counts.size else 1)
    ax.set_xlim(-maximum * 1.2, maximum * 1.2)
    ax.axvline(0, color="#555555", linewidth=0.8)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=7))
    ax.xaxis.set_major_formatter(FuncFormatter(lambda value, position: f"{abs(value):,.0f}"))
    ax.set_title(f"{occupation} (shown n = {int(counts.to_numpy().sum())})")
    ax.set_xlabel("Observed people (absolute counts); home area bins in m²")
    ax.grid(axis="x", alpha=0.2)
    ax.legend(fontsize=8)
for ax in list(axes.flat)[len(home_occupation_order):]:
    ax.set_visible(False)
fig.suptitle("Home-area and sex pyramids within occupation groups", fontsize=15)
plt.show()
plt.close(fig)
print(f"Stacked histogram omits {int(home.isna().sum())} missing/non-numeric home areas.")
print(f"Faceted pyramids omit {int((home.isna() | other_sex).sum())} rows with missing home area or an unplotted sex label.")
display(pd.crosstab(home_bins, home_occupation, dropna=False).reindex(index=home_bin_labels, fill_value=0))

### Monthly wage versus wealth

This scatterplot uses two continuous measurements with their own units. Each point represents a row with both values observed. Sex is a categorical color label, never a numeric code. Pearson correlation summarizes linear association and is sensitive to extreme values; Spearman correlation summarizes rank association. Neither statistic makes a causal claim, and their patterns partly reflect the simulation design.

In [ ]:
paired = raw_numeric[["monthly_wage_try", "wealth_try"]].dropna()
fig, ax = plt.subplots(figsize=(11, 7), constrained_layout=True)
palette = {"Kadın": "#a04e89", "Erkek": "#447e9e"}
for label in sex_labels.loc[paired.index].unique():
    group_index = paired.index[sex_labels.loc[paired.index].eq(label)]
    ax.scatter(paired.loc[group_index, "monthly_wage_try"], paired.loc[group_index, "wealth_try"],
               s=24, alpha=0.55, color=palette.get(label, "#777777"), label=label)
ax.set_title(f"Monthly wage versus wealth — {len(paired):,} complete pairs")
ax.set_xlabel("Monthly wage (TRY/month)")
ax.set_ylabel("Wealth (TRY)")
ax.xaxis.set_major_formatter(number_formatter)
ax.yaxis.set_major_formatter(number_formatter)
ax.grid(alpha=0.2)
if len(paired):
    ax.legend(title="Sex", fontsize=9)
plt.show()
plt.close(fig)
print(f"Pairwise analysis omits {len(df) - len(paired):,} rows missing either numeric value.")
if len(paired) >= 2 and paired.nunique().min() > 1:
    print(f"Pearson r: {paired.corr(method='pearson').iloc[0, 1]:.3f}")
    print(f"Spearman rho: {paired.corr(method='spearman').iloc[0, 1]:.3f}")
else:
    print("Correlation is undefined: fewer than two complete pairs or a constant variable.")
print("These are descriptive associations in synthetic data, not evidence that wage changes cause wealth changes.")

## 5. Data quality and preprocessing

Keep an untouched `df_raw` and make all changes in `df_clean`. First inspect missingness, column concentration and IQR flags. Then exclude the record with the impossible schooling measurement **before fitting any imputer**; otherwise its value of 1,000 years could distort distances and scaling. Finally impute only the requested variables.

An IQR flag is a prompt to investigate, not proof of an error. High wealth can be real, and 25 years of education can be plausible for a person with advanced postgraduate study. Neither is removed merely for lying in a distribution's tail.

This is exploratory preparation of a synthetic dataset. If these transformations later feed a predictive model, split training and test data first and fit every imputer and scaler on the training partition only.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

df_raw = df.copy(deep=True)
assert len(df_raw) == 554, "This exercise expects 554 synthetic persons."

def missingness_summary(frame):
    missing_count = frame.isna().sum()
    return pd.DataFrame({
        "dtype": frame.dtypes.astype(str),
        "nonmissing_count": frame.notna().sum(),
        "missing_count": missing_count,
        "missing_pct": (100 * missing_count / len(frame)).round(2),
    }).rename_axis("column")

missing_analysis = missingness_summary(df_raw)
display(missing_analysis)
print("Missing values are counted for every column, including identifiers and categories.")


### Columns dominated by one value

The denominator below is **all 554 records**, not just complete observations. Missing observations form their own explicit bucket. A column is flagged when either an observed value or that missing bucket occupies **strictly more than 75%** of rows. The table also shows the most common observed value separately, so a missingness problem is not mistaken for an ordinary constant feature.

Review such columns for usefulness and collection problems. A flag does not automatically justify deleting a column, and high-cardinality identifiers or names are not removed by this check.

In [ ]:
def concentration_summary(frame, threshold=0.75):
    records = []
    total = len(frame)
    for column in frame.columns:
        observed_counts = frame[column].value_counts(dropna=True)
        missing_count = int(frame[column].isna().sum())
        observed_count = int(observed_counts.iloc[0]) if len(observed_counts) else 0
        observed_value = str(observed_counts.index[0]) if len(observed_counts) else "<none>"
        if missing_count > observed_count:
            dominant_value = "<Missing>"
            dominant_count = missing_count
        else:
            dominant_value = observed_value
            dominant_count = observed_count
        records.append({
            "column": column,
            "distinct_observed_values": int(frame[column].nunique(dropna=True)),
            "most_common_observed_value": observed_value,
            "most_common_observed_count": observed_count,
            "most_common_observed_pct_of_all_rows": round(100 * observed_count / total, 2),
            "missing_pct_of_all_rows": round(100 * missing_count / total, 2),
            "dominant_bucket_including_missing": dominant_value,
            "dominant_bucket_count": dominant_count,
            "dominant_bucket_pct": round(100 * dominant_count / total, 2),
            "more_than_75pct_identical": dominant_count / total > threshold,
        })
    return pd.DataFrame(records).set_index("column")

concentration_analysis = concentration_summary(df_raw)
display(concentration_analysis)
dominant_columns = concentration_analysis.index[
    concentration_analysis["more_than_75pct_identical"]
].tolist()
print("Columns exceeding 75% concentration:", dominant_columns or "None")


### IQR screening and contextual review

For each numeric column, the usual screening interval is \(Q_1 - 1.5\,IQR\) to \(Q_3 + 1.5\,IQR\), where \(IQR = Q_3-Q_1\). Missing values are excluded when estimating quartiles. Compare these statistical flags with the meaning and units of the variable: wealth and floor area often have long right tails, while schooling reported as 1,000 literal years is impossible.

The conservative schooling-error screen below selects negative values or values above 100 years. The age comparison is also displayed as a contextual review flag; it is not, by itself, an automatic deletion rule. This protects plausible 25-year postgraduate values from being classified as errors simply because they are unusual.

In [ ]:
def iqr_screen(frame, columns):
    records = []
    flags = pd.DataFrame(False, index=frame.index, columns=columns)
    for column in columns:
        values = pd.to_numeric(frame[column], errors="raise")
        observed = values.dropna()
        q1 = observed.quantile(0.25)
        q3 = observed.quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        column_flags = ((values < lower) | (values > upper)).fillna(False)
        flags[column] = column_flags
        records.append({
            "column": column,
            "observed_count": int(observed.size),
            "q1": q1,
            "q3": q3,
            "iqr": iqr,
            "lower_fence": lower,
            "upper_fence": upper,
            "below_fence_count": int((values < lower).sum()),
            "above_fence_count": int((values > upper).sum()),
            "iqr_flag_count": int(column_flags.sum()),
        })
    return pd.DataFrame(records).set_index("column"), flags

raw_iqr_summary, raw_iqr_flags = iqr_screen(df_raw, numeric_cols)
display(raw_iqr_summary.round(2))
print("Records with at least one IQR flag:", int(raw_iqr_flags.any(axis=1).sum()))

schooling = pd.to_numeric(df_raw["education_years"], errors="raise")
schooling_context_flags = pd.DataFrame({
    "negative_schooling": schooling.lt(0),
    "schooling_above_100_years": schooling.gt(100),
    "schooling_exceeds_age_review": schooling.gt(df_raw["age"]),
    "unusual_but_potentially_plausible_25_years": schooling.eq(25),
}, index=df_raw.index)
measurement_error_mask = (
    schooling_context_flags["negative_schooling"]
    | schooling_context_flags["schooling_above_100_years"]
)
measurement_error_rows = df_raw.loc[measurement_error_mask].copy()
review_columns = ["person_id", "age", "education_years", "education_level",
                  "university_completed", "department_completed"]
display(df_raw.loc[schooling_context_flags.any(axis=1), review_columns]
        .join(schooling_context_flags.loc[schooling_context_flags.any(axis=1)]))
print("Records excluded for impossible schooling measurements:", len(measurement_error_rows))
assert len(measurement_error_rows) == 1, "Expected one impossible schooling record in this exercise."
assert measurement_error_rows["education_years"].eq(1000).all()

# The raw data and the excluded record remain available for audit.
df_clean = df_raw.loc[~measurement_error_mask].copy(deep=True)
assert len(df_clean) == len(df_raw) - len(measurement_error_rows)
assert df_clean["education_years"].eq(25).sum() == df_raw["education_years"].eq(25).sum()
print(f"Retained {len(df_clean)} persons; no record was removed for an IQR flag alone.")


**Why exclude the whole record?** If independent source information could recover the schooling value, correcting that value would preserve the rest of the record. Another option is to mark schooling as missing and impute it. Here the exercise selects exclusion of the affected record: there is no trusted replacement, and removing it avoids fitting distance-based preprocessing on a corrupted measurement. The original row is retained in `df_raw` and `measurement_error_rows`; this is one excluded person out of 554, not general deletion of statistical outliers.

### Requested imputations

* **Wealth and monthly wage:** use the median of the remaining observed values. The median is less affected by legitimate extreme financial values than the mean. Wage is in TRY per month; wealth is in TRY.
* **Height and weight:** use the mean. These body measurements are comparatively concentrated in this synthetic exercise; a mean is a reasonable simple baseline after contextual review. It is still an approximation and reduces variation at the filled positions.
* **Home area:** use five-neighbour KNN with standardized numeric predictors. KNN is sensitive to scale: an unscaled TRY variable would overwhelm years, centimetres and kilograms. Use `log1p(wealth)` as a distance feature to reduce the dominance of the wealth tail.

Only missing entries are filled. Other missing categorical fields remain missing, and the notebook reports them explicitly. These simple imputations provide a complete set of the requested numeric fields; they do not establish that missingness is random or quantify uncertainty.

In [ ]:
missing_before_imputation = df_clean.isna().sum()
wealth_observed_before_imputation = df_clean["wealth_try"].dropna().copy()
requested_imputation_columns = [
    "wealth_try", "monthly_wage_try", "height_cm", "weight_kg", "home_sqm"
]
observed_requested_values = {
    column: df_clean.loc[df_clean[column].notna(), column].copy()
    for column in requested_imputation_columns
}
imputation_records = []

for column, strategy in [
    ("wealth_try", "median"),
    ("monthly_wage_try", "median"),
    ("height_cm", "mean"),
    ("weight_kg", "mean"),
]:
    observed = df_clean[column].dropna()
    assert not observed.empty, f"Cannot estimate an imputation value for {column}."
    fill_value = observed.median() if strategy == "median" else observed.mean()
    missing_mask = df_clean[column].isna()
    df_clean.loc[missing_mask, column] = fill_value
    imputation_records.append({
        "column": column,
        "method": strategy,
        "observed_donors": int(observed.size),
        "filled_count": int(missing_mask.sum()),
        "constant_fill_value": float(fill_value),
    })

display(pd.DataFrame(imputation_records).set_index("column").round(3))


The KNN predictors are age, height, weight, education years, monthly wage and log-transformed wealth. Earlier mean/median fills supply complete body and financial predictors. The impossible schooling record has already been excluded. `StandardScaler` estimates each feature's centre and spread from available observations; its NaN support preserves the missing home areas.

For a missing home area, the home-area coordinate is absent from the distance calculation, so neighbours are selected using the available standardized predictors. The target is scaled too for convenient inverse transformation; it does not contribute to distances for these missing-target rows. The imputer's output is copied back **only into originally missing `home_sqm` entries**. Observed areas and every other field are preserved. Neighbour-based estimates are heuristic and depend on predictor choice, five-neighbour smoothing and the synthetic population; they are not independent measurements of floor area.

In [ ]:
home_missing_mask = df_clean["home_sqm"].isna()
assert df_clean["wealth_try"].ge(0).all(), "log1p wealth requires nonnegative wealth in this exercise."
knn_features = df_clean[[
    "age", "height_cm", "weight_kg", "education_years", "monthly_wage_try"
]].copy()
knn_features["log1p_wealth_try"] = np.log1p(df_clean["wealth_try"])
knn_features["home_sqm"] = df_clean["home_sqm"]
assert knn_features.drop(columns="home_sqm").notna().all().all(), "Review missing KNN predictors."
assert knn_features["home_sqm"].notna().sum() >= 5, "KNN needs at least five observed home-area donors."

knn_scaler = StandardScaler()
standardized_features = knn_scaler.fit_transform(knn_features)
home_imputer = KNNImputer(n_neighbors=5, weights="uniform", metric="nan_euclidean")
standardized_imputed = home_imputer.fit_transform(standardized_features)
restored_features = pd.DataFrame(
    knn_scaler.inverse_transform(standardized_imputed),
    index=knn_features.index,
    columns=knn_features.columns,
)
df_clean.loc[home_missing_mask, "home_sqm"] = restored_features.loc[home_missing_mask, "home_sqm"]
imputation_records.append({
    "column": "home_sqm",
    "method": "KNN, k=5, standardized predictors including log1p wealth",
    "observed_donors": int((~home_missing_mask).sum()),
    "filled_count": int(home_missing_mask.sum()),
    "constant_fill_value": np.nan,
})

print("Home areas imputed:", int(home_missing_mask.sum()))
display(df_clean.loc[home_missing_mask, ["person_id", "home_sqm"]].head(10).round(2))

# Meaningful checks: retain every observed requested value and fill all requested gaps.
for column, observed in observed_requested_values.items():
    pd.testing.assert_series_equal(df_clean.loc[observed.index, column], observed)
assert df_clean[requested_imputation_columns].notna().all().all()
assert df_clean["home_sqm"].gt(0).all()
pd.testing.assert_frame_equal(df, df_raw)
print("All requested numeric gaps filled; observed values and the raw source are unchanged.")


### Wealth: raw scale and logarithmic scale

Compare the **same observed, nonnegative wealth values** on the raw scale and after `log1p`, which also supports a zero wealth value. The transform usually compresses a long right tail and reduces skewness, making the distribution and distance comparisons easier to examine. Check the displayed skewness values to judge the effect in this sample. It changes interpretation to a logarithmic scale; it does not prove normality, erase valid wealthy individuals, or replace the TRY wealth column in the cleaned data.

In [ ]:
wealth_for_transform = wealth_observed_before_imputation
assert wealth_for_transform.ge(0).all()
wealth_log1p = np.log1p(wealth_for_transform)
wealth_skewness = pd.DataFrame({
    "scale": ["Observed wealth, TRY", "log1p(observed wealth, TRY)"],
    "observation_count": [len(wealth_for_transform), len(wealth_log1p)],
    "sample_skewness": [wealth_for_transform.skew(), wealth_log1p.skew()],
})
display(wealth_skewness.round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(wealth_for_transform, bins=35, color="#2171b5", edgecolor="white")
axes[0].set(title="Observed wealth: raw scale", xlabel="Wealth (TRY)", ylabel="Persons")
axes[0].ticklabel_format(axis="x", style="sci", scilimits=(0, 0))
axes[1].hist(wealth_log1p, bins=35, color="#238b45", edgecolor="white")
axes[1].set(title="The same observations: log1p scale", xlabel="log(1 + wealth in TRY)", ylabel="Persons")
fig.suptitle("Wealth transformation: valid tails are retained")
fig.tight_layout()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(OUTPUT_DIR / "Week2_EDA_wealth_transform.png", dpi=160, bbox_inches="tight")
plt.show()


### Audit the result and save the cleaned dataset

The final report compares missingness before and after the requested imputations. Any remaining missing categories are shown rather than silently filled. IQR screening is repeated on the cleaned numeric data as a descriptive check; its remaining flags are retained. The CSV uses the original columns and units, with wealth stored in TRY. The accompanying quality report and metadata record the exclusion and imputation decisions.

In [ ]:
quality_summary = missing_analysis.copy()
quality_summary = quality_summary.rename(columns={
    "nonmissing_count": "raw_nonmissing_count",
    "missing_count": "raw_missing_count",
    "missing_pct": "raw_missing_pct",
})
quality_summary["missing_after_record_exclusion"] = missing_before_imputation
quality_summary["missing_after_imputation"] = df_clean.isna().sum()
quality_summary["clean_missing_pct"] = (100 * df_clean.isna().sum() / len(df_clean)).round(2)
quality_summary["values_imputed"] = missing_before_imputation - df_clean.isna().sum()
quality_summary["imputation_method"] = "not requested; preserved"
for record in imputation_records:
    quality_summary.loc[record["column"], "imputation_method"] = record["method"]
quality_summary["raw_more_than_75pct_identical"] = concentration_analysis["more_than_75pct_identical"]
display(quality_summary)

remaining_missing = df_clean.isna().sum()
print("Remaining missing fields:")
display(remaining_missing[remaining_missing.gt(0)].rename("missing_count").to_frame())

clean_iqr_summary, clean_iqr_flags = iqr_screen(df_clean, numeric_cols)
display(clean_iqr_summary.round(2))
print("Cleaned records with an IQR flag (retained):", int(clean_iqr_flags.any(axis=1).sum()))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
cleaned_csv_path = OUTPUT_DIR / "Week2_EDA_cleaned.csv"
quality_csv_path = OUTPUT_DIR / "Week2_EDA_quality_summary.csv"
metadata_path = OUTPUT_DIR / "Week2_EDA_preprocessing_metadata.json"
df_clean.to_csv(cleaned_csv_path, index=False)
quality_summary.to_csv(quality_csv_path, index=True)
preprocessing_metadata = {
    "data_kind": "synthetic educational exercise; not real respondents",
    "raw_rows": int(len(df_raw)),
    "clean_rows": int(len(df_clean)),
    "excluded_measurement_error_rows": int(measurement_error_mask.sum()),
    "measurement_error_rule": "education_years < 0 or education_years > 100",
    "selected_correction": "exclude affected whole record; preserve raw data and excluded rows in memory",
    "iqr_rule": "Q1 - 1.5*IQR and Q3 + 1.5*IQR; descriptive flags, no automatic removal",
    "dominance_rule": "strictly more than 75% of all raw rows; missing is an explicit bucket",
    "knn_neighbors": 5,
    "knn_predictors": knn_features.columns.drop("home_sqm").tolist(),
    "knn_scaling": "StandardScaler; target omitted from distances when missing; inverse transform target output",
    "imputations": [
        {key: (None if key == "constant_fill_value" and pd.isna(value) else value)
         for key, value in record.items()}
        for record in imputation_records
    ],
    "remaining_missing": {column: int(count) for column, count in remaining_missing.items() if count > 0},
    "financial_units": {"monthly_wage_try": "TRY/month", "wealth_try": "TRY"},
    "observed_requested_values_preserved": True,
    "raw_source_unchanged": True,
}
metadata_path.write_text(json.dumps(preprocessing_metadata, indent=2, ensure_ascii=False), encoding="utf-8")
print("Saved:", cleaned_csv_path, quality_csv_path, metadata_path, sep="\n")


## Discussion and next practice

1. Why is a 1,000-year education value a measurement error, while a large wealth value may be valid?
2. How do means and medians respond differently to a long upper tail?
3. Why standardize predictors before nearest-neighbor imputation?
4. Which associations would need further evidence before a causal interpretation?
5. What changes when missingness depends on unobserved income rather than being placed for a classroom example?

Restart the runtime and run all cells before sharing your work. Preserve the raw CSV and compare it with the exported cleaned version.

Reference documentation: [pandas](https://pandas.pydata.org/docs/), [seaborn](https://seaborn.pydata.org/), [scikit-learn imputation](https://scikit-learn.org/stable/modules/impute.html), [Google Colab](https://research.google.com/colaboratory/faq.html).